# HW2 - Build a "Silver" flat view (about 30 min)
**Slides 4, 15, 17, 25.** Bronze keeps the data as it came (schema on read). Silver enforces names and types (schema on write, slide 4).
You will build two Silver tables in plain Python (dev_env has no pandas, and you do not need it):

- `orders_flat`: **one row per order** with a unified email, a numeric total, a lowercase status and a real `datetime` for `created_at`
- `order_items`: **one row per item** with `order_id`, `item_idx`, `sku`, `qty` (int) and `unit_price`

**Before you start:** `bash lab/reset.sh` from the repo folder on your laptop. Fill in the `# TODO` cells, then run the self-checks.

In [ ]:
import os
from pymongo import MongoClient

# Inside dev_env the host is "mongo" (Jupyter does not see $MONGO_URI, so we fall back to the same value).
URI = os.getenv("MONGO_URI", "mongodb://mongo:27017/?directConnection=true")
client = MongoClient(URI, serverSelectionTimeoutMS=5000)
db = client["shop"]
coll = db["orders"]
print("connected to", URI, "| orders:", coll.count_documents({}))

docs = list(coll.find())
print("loaded", len(docs), "documents")

## 1. Helpers: one function per messy field
- **email** (slide 15): v1 has `customer.email`, v2 has `customer.contact.email`
- **total** (slide 11): `Decimal128`, `float`, `str` or missing -> a `float`, or `None` when missing
- **status**: lowercase, `None` when missing
- **created_at** (slide 11): BSON Date (`datetime`), ISO string like `"2026-03-31T15:06:00Z"`, or epoch **milliseconds** (`int`) -> a `datetime` in UTC

PyMongo returns BSON Dates as *naive* UTC datetimes. Return **naive UTC** datetimes for all three formats so they compare cleanly.

In [ ]:
from datetime import datetime, timezone
from bson import Decimal128

def unified_email(doc):
    # TODO
    raise NotImplementedError

def total_as_number(doc):
    # TODO: float or None
    raise NotImplementedError

def clean_status(doc):
    # TODO: lowercase string or None
    raise NotImplementedError

def created_at_as_datetime(doc):
    # TODO: naive UTC datetime for all three formats
    raise NotImplementedError

## 2. Build `orders_flat` (one row per order)
Each row is a dict with keys: `_id`, `order_id`, `schema_v`, `email`, `total`, `status`, `created_at`, `city`.
Keep `_id` - it is the MERGE key downstream (`order_id` has duplicates!).

In [ ]:
orders_flat = []
# TODO: one dict per document in `docs`


In [ ]:
print(len(orders_flat), "rows")
print(orders_flat[:1])

## 3. Build `order_items` (one row per item, slides 17 and 25)
Each row: `order_id`, `item_idx` (position in the array), `sku`, `qty` (**int** - some are strings like `"2"`), `unit_price`.

**Decide:** what happens to the 40 orders with an empty `items` array?
- `KEEP_EMPTY_ORDERS = True` -> keep one row per empty order with `None` item fields (like `explode_outer`)
- `KEEP_EMPTY_ORDERS = False` -> drop them (like `explode` / `$unwind` without options)

In [ ]:
KEEP_EMPTY_ORDERS = None   # TODO: True or False - and explain your choice in the markdown cell below

order_items = []
# TODO: one dict per item (and per empty order if KEEP_EMPTY_ORDERS)


**My choice for empty orders and why (2-3 sentences):**

*TODO: write here. Think about who reads `order_items` and whether an order with no items is still visible in `orders_flat`.*

## Self-checks (do not edit)

In [ ]:
from collections import Counter

# orders_flat
assert len(orders_flat) == 200, f"expected 200 rows (one per document), got {len(orders_flat)}"
print("OK orders_flat has 200 rows")
assert sum(r["email"] is None for r in orders_flat) == 0, "some emails are None - check the v2 location"
print("OK zero null emails")
assert all(isinstance(r["total"], float) or r["total"] is None for r in orders_flat), "total must be float or None"
assert sum(r["total"] is None for r in orders_flat) == 11, "exactly 11 orders have no total"
print("OK total is a number (11 missing)")
assert {r["status"] for r in orders_flat} == {"created", "paid", "shipped", "delivered", "cancelled", None}
assert sum(r["status"] is None for r in orders_flat) == 7
print("OK status is lowercase (7 missing)")
assert all(isinstance(r["created_at"], datetime) and r["created_at"].tzinfo is None for r in orders_flat), \
    "created_at must be a naive UTC datetime for every row"
assert all(r["created_at"].year == 2026 for r in orders_flat), "dates outside 2026 - check the epoch-millis conversion"
print("OK created_at is a datetime for all 200 rows")

# order_items
assert KEEP_EMPTY_ORDERS in (True, False), "set KEEP_EMPTY_ORDERS to True or False"
expected_rows = 381 if KEEP_EMPTY_ORDERS else 341
assert len(order_items) == expected_rows, f"expected {expected_rows} item rows, got {len(order_items)}"
print(f"OK order_items has {expected_rows} rows (KEEP_EMPTY_ORDERS={KEEP_EMPTY_ORDERS})")
real_items = [r for r in order_items if r["item_idx"] is not None]
assert all(type(r["qty"]) is int for r in real_items), "qty must be int"
assert sum(r["qty"] for r in real_items) == 682, "total quantity should be 682"
print("OK qty is int (total quantity 682)")
print("HW2 self-check passed")

**Reflection (1-2 sentences):** `order_id` is not unique (ORD-10010, ORD-10011). Which key would you use to MERGE `orders_flat` into the warehouse, and why?